## Background Research

**What is HotpotQA?**
HotpotQA is a QA dataset where most questions require reasoning across *two*
Wikipedia paragraphs, not one. Each example includes "supporting facts" —
sentence-level annotations showing exactly which sentences, in which
paragraphs, were needed to answer. This makes it a good testbed for multi-hop
reasoning, and a good stress-test for standard RAG, which is built around
single retrieve-then-generate.

**Why single-shot RAG struggles here**
A standard RAG pipeline embeds the question once, retrieves top-k passages
once, and generates an answer once. If the answer requires a fact from
document A ("Which team did X play for") to know what to search for in
document B ("What year did that team win the championship"), a single
retrieval pass over the *original* question will often miss document B
entirely — the question text never mentions the team name.

**The agentic alternative**
An agent with tool-use can decide, after seeing the first retrieval, that it
needs another search — using an intermediate fact it just learned as the next
query. This turns retrieval from a fixed one-shot step into a controlled loop:
retrieve → reason → decide (answer or retrieve again) → repeat.

**Plan for this project**
1. Build a baseline: single-retrieval RAG on HotpotQA (expected to do
   reasonably on single-hop-friendly questions, poorly on genuine multi-hop
   ones — this is the hypothesis to test, not an assumed result).
2. Build the agentic version: Claude with a `search_documents` tool it can
   call repeatedly, deciding for itself when it has enough evidence.
3. Score both with exact-match and F1 against HotpotQA's gold answers, on the
   same held-out slice, so the comparison is fair.
4. Read through the *actual* transcripts afterward and write up specific
   cases — including at least one clear failure — rather than only reporting
   the aggregate number.

Nothing below this cell is written yet. Numbers get filled in only after code
runs and produces them.

In [2]:
import sys
from pathlib import Path

# Add the repo root (parent of notebooks/) to sys.path so `src` is importable
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

In [4]:
from src.data import load_hotpotqa

data = load_hotpotqa()
print(f"Loaded {len(data)} examples")
example = data[0]
print("Question:", example["question"])
print("Answer:", example["answer"])
print("Num supporting facts:", len(example["supporting_facts"]["sent_id"]))
print("Num context docs:", len(example["context"]["title"]))

c:\Users\edrin\OneDrive\Desktop\Self Initiated Projects\Agentic-rag-research-assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 200 examples
Question: What nationality was Oliver Reed's character in the film Royal Flash?
Answer: Prussian
Num supporting facts: 3
Num context docs: 10


## Data Exploration

Loaded a 200-example slice of HotpotQA (distractor config, validation split,
seed=42) via the Hugging Face `datasets` library.

**Note on dataset source:** the original `hotpot_qa` repo used a Python
loading script, which is no longer supported by recent versions of the
`datasets` library (`datasets>=4.0` dropped script-based loading entirely).
The maintainers have since published a Parquet-format version under
`hotpotqa/hotpot_qa`, which this project uses instead — same data, standard
columnar format, no `trust_remote_code` needed.

**First example, to sanity-check the schema:**
- Question: *"What nationality was Oliver Reed's character in the film Royal Flash?"*
- Answer: `Prussian`
- Supporting facts: 3 sentences
- Context documents: 10

This confirms the "distractor" setup in practice: the model is given 10
context documents, but only a subset of sentences within 2 of them are
actually needed (3 supporting facts) — the rest are plausible-looking
distractors. This is exactly the setup that should punish naive single-shot
RAG if it grabs the wrong subset of documents on the first retrieval pass,
and is the property the baseline-vs-agentic comparison in this project is
designed to test.

**Next step:** build the baseline single-retrieval RAG pipeline first, per
the plan above, and score it before touching the agentic version — so the
"does the agent actually help" question has a real number to compare against
rather than an assumed one.